<a href="https://colab.research.google.com/github/romandudarets/Data-Analysis-University/blob/main/%D0%9B%D1%809_%D0%94%D1%83%D0%B4%D0%B0%D1%80%D0%B5%D1%86%D1%8C_4_9_ipynb%22.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Лабораторна робота № 9
## Рекомендаційні системи: SVD та KNNBasic

**ПІБ студента:** Дударець Роман Богданович
**Група:** 4-9  
**Дата:** 22.11.2026

> **Призначення:** шаблон для самостійного виконання ЛР на основі прикладу з алгоритмами `SVD` і `KNNBasic` у бібліотеці `Surprise`. Комірки з позначкою **TODO** потрібно заповнити власним кодом. Це не готовий розв’язок і не заміна офіційної умови роботи.

**Мета:** завантажити дані про оцінки фільмів, побудувати дві рекомендаційні моделі, підібрати гіперпараметри, порівняти моделі за RMSE/MAE та сформувати топ-10 рекомендацій для користувача.

**Дані:** вбудований набір `MovieLens 100K` (`ml-100k`) із бібліотеки `Surprise`.

**Послідовність роботи:** підготовка середовища → дані → розбиття → налаштування моделей → навчання → тестування → рекомендації → висновки.

## 1. Підготовка середовища

Комбінація бібліотек для **Google Colab із Python 3.13**. Встановлення без `--force-reinstall`, щоб без потреби не перевстановлювати всі залежності. Після зміни NumPy чи SciPy у вже запущеному середовищі **перезапустіть сесію** (`Runtime → Restart session`), а потім виконайте комірки нижче з початку.

In [ ]:
%pip install -q \
    numpy==2.2.6 \
    scipy==1.15.3 \
    scikit-learn==1.6.1 \
    pandas==2.2.3 \
    scikit-surprise==1.1.5

In [ ]:
import sys
import numpy as np
import scipy
import sklearn
import pandas as pd
import surprise

from surprise import Dataset, Reader, SVD, KNNBasic, accuracy
from surprise.model_selection import GridSearchCV
from sklearn.model_selection import train_test_split as sklearn_train_test_split

print('Python:', sys.version.split()[0])
print('NumPy:', np.__version__)
print('SciPy:', scipy.__version__)
print('scikit-learn:', sklearn.__version__)
print('pandas:', pd.__version__)
print('Surprise:', surprise.__version__)

Python: 3.13.18
NumPy: 2.4.6
SciPy: 1.15.3
scikit-learn: 1.6.1
pandas: 2.2.3
Surprise: 1.1.5


## 2. Завантаження та первинний аналіз даних

1. Завантажте вбудований набір `ml-100k`.
2. Перетворіть оцінки у `pandas.DataFrame` з колонками `user_id`, `item_id`, `rating`, `timestamp`.
3. Виведіть розмір таблиці, декілька рядків, кількість унікальних користувачів та фільмів, розподіл оцінок.

**Зверніть увагу:** ідентифікатори користувачів і фільмів у `Surprise` є зовнішніми (raw) ID; вони можуть мати рядковий тип.

In [ ]:
# Завантаження даних (готова частина шаблону)
data = Dataset.load_builtin('ml-100k', prompt=False)

ratings_df = pd.DataFrame(
    data.raw_ratings,
    columns=['user_id', 'item_id', 'rating', 'timestamp']
)

print('Розмір:', ratings_df.shape)
display(ratings_df.head())

Trying to download dataset from https://files.grouplens.org/datasets/movielens/ml-100k.zip...
Done! Dataset ml-100k has been saved to /root/.surprise_data/ml-100k
Розмір: (100000, 4)


,user_id,item_id,rating,timestamp
0,196,242,3.0,881250949
1,186,302,3.0,891717742
2,22,377,1.0,878887116
3,244,51,2.0,880606923
4,166,346,1.0,886397596


In [ ]:
print("Кількість унікальних користувачів:", ratings_df["user_id"].nunique())
print("Кількість унікальних фільмів:", ratings_df["item_id"].nunique())
print("Мінімальна оцінка:", ratings_df["rating"].min())
print("Максимальна оцінка:", ratings_df["rating"].max())
print("Середня оцінка:", round(ratings_df["rating"].mean(), 4))
print("\nПропущені значення:\n", ratings_df.isnull().sum())
print("\nРозподіл оцінок:\n", ratings_df["rating"].value_counts().sort_index())

Кількість унікальних користувачів: 943
Кількість унікальних фільмів: 1682
Мінімальна оцінка: 1.0
Максимальна оцінка: 5.0
Середня оцінка: 3.5299

Пропущені значення:
 user_id      0
item_id      0
rating       0
timestamp    0
dtype: int64

Розподіл оцінок:
 rating
1.0     6110
2.0    11370
3.0    27145
4.0    34174
5.0    21201
Name: count, dtype: int64


**Короткі спостереження щодо даних:**

Набір даних ml-100k містить 100 000 оцінок від 943 користувачів для 1 682 фільмів без пропущених значень. Оцінки задані за шкалою від 1 до 5 із середнім значенням близько 3.53, при цьому найчастіше користувачі ставлять 3 та 4 бали.

## 3. Розбиття на навчальну та тестову вибірки

Виконайте випадкове розбиття **80% / 20%** із `random_state=42` за прикладом. Для налаштування гіперпараметрів використовуйте **лише навчальну** частину. Тестова вибірка має залишатися недоторканою до фінальної оцінки.

In [ ]:
train_df, test_df = sklearn_train_test_split(
    ratings_df, test_size=0.2, random_state=42
)

print("Розмір train_df:", train_df.shape)
print("Розмір test_df: ", test_df.shape)

Розмір train_df: (80000, 4)
Розмір test_df:  (20000, 4)


In [ ]:
reader = Reader(rating_scale=data.reader.rating_scale)
train_data = Dataset.load_from_df(
    train_df[["user_id", "item_id", "rating"]], reader
)
testset = list(
    test_df[["user_id", "item_id", "rating"]].itertuples(index=False, name=None)
)

print("Кількість записів у train_df:", len(train_df))
print("Кількість записів у testset: ", len(testset))

Кількість записів у train_df: 80000
Кількість записів у testset:  20000


## 4. Гіперпараметри моделей

Для `SVD` розгляньте кількість прихованих факторів, швидкість навчання та регуляризацію. Для `KNNBasic` — число сусідів, мінімальне число сусідів та метрику схожості.

Нижче — **початкова сітка** з аналізованого прикладу. Її можна змінити й обґрунтувати вибір.

In [ ]:
# Стартові сітки параметрів (можна змінювати)
param_grid_svd = {
    'n_factors': [50, 100],
    'lr_all': [0.005, 0.010],
    'reg_all': [0.02, 0.05]
}

param_grid_knn = {
    'k': [20, 40],
    'min_k': [1, 5],
    'sim_options': {
        'name': ['cosine', 'pearson'],
        'user_based': [False, True]
    }
}

**Обґрунтування вибору параметрів:**

Для SVD ключовими є кількість прихованих факторів (n_factors — визначає місткість латентного простору), швидкість навчання (lr_all — впливає на збіжність градієнтного спуску) та коефіцієнт регуляризації (reg_all — запобігає перенавчанню). Для KNNBasic важливими є число найближчих сусідів (k), мінімальне число сусідів (min_k для стабільності оцінок) і параметр sim_options (name: 'cosine' або 'pearson', user_based: True або False — колаборативна фільтрація за користувачами чи за об'єктами).

## 5. Пошук найкращих гіперпараметрів (GridSearchCV)

Для обох моделей використайте `GridSearchCV` з метриками `rmse` та `mae` і `cv=3`. Навчайте об’єкти пошуку **лише на `train_data`**.

In [ ]:
grid_search_svd = GridSearchCV(
    SVD,
    param_grid_svd,
    measures=["rmse", "mae"],
    cv=3,
    n_jobs=-1,
)
grid_search_svd.fit(train_data)

In [ ]:
grid_search_knn = GridSearchCV(
    KNNBasic,
    param_grid_knn,
    measures=["rmse", "mae"],
    cv=3,
    n_jobs=-1,
)
grid_search_knn.fit(train_data)

In [ ]:
print("=== SVD (CV) ===")
print("Найкращий CV RMSE:", round(grid_search_svd.best_score["rmse"], 4))
print("Найкращий CV MAE: ", round(grid_search_svd.best_score["mae"], 4))
print("Найкращі параметри за RMSE:", grid_search_svd.best_params["rmse"])

print("\n=== KNNBasic (CV) ===")
print("Найкращий CV RMSE:", round(grid_search_knn.best_score["rmse"], 4))
print("Найкращий CV MAE: ", round(grid_search_knn.best_score["mae"], 4))
print("Найкращі параметри за RMSE:", grid_search_knn.best_params["rmse"])

**Аналіз результатів кросвалідації:**

| Модель | Найкращий CV RMSE | CV MAE | Найкращі параметри за RMSE |
|:--|--:|--:|:--|
| SVD | ... | ... | ... |
| KNNBasic | ... | ... | ... |

На перехресній перевірці (cv=3) матрична факторизація SVD зазвичай показує нижчі помилки (RMSE та MAE), ніж базовий метод найближчих сусідів KNNBasic, оскільки краще узагальнює приховані патерни в розрідженій матриці оцінок. Найкращі параметри для кожної моделі зафіксовано у виводі вище (grid_search_svd.best_params['rmse'] та grid_search_knn.best_params['rmse']).

## 6. Навчання остаточних моделей

Побудуйте `trainset = train_data.build_full_trainset()`. Створіть нові екземпляри SVD і KNNBasic з параметрами, обраними за **CV RMSE**, після чого навчіть їх на **всій навчальній вибірці**.

In [11]:
best_svd_params = grid_search_svd.best_params["rmse"]
best_knn_params = grid_search_knn.best_params["rmse"]

trainset = train_data.build_full_trainset()

best_svd = SVD(**best_svd_params, random_state=42)
best_svd.fit(trainset)

best_knn = KNNBasic(**best_knn_params)
best_knn.fit(trainset)

print("Обидві фінальні моделі успішно навчено на повному trainset.")

Computing the cosine similarity matrix...
Done computing similarity matrix.
Обидві фінальні моделі успішно навчено на повному trainset.


## 7. Оцінювання на незалежній тестовій вибірці

Отримайте передбачення для `testset` методами `.test()` обох моделей. Обчисліть `RMSE` та `MAE` за допомогою `surprise.accuracy`.

**Методична примітка:** вибирайте модель для рекомендацій за результатами **кросвалідації**, а незалежну тестову вибірку використовуйте для фінальної оцінки якості, а не для повторного підбору.

In [12]:
svd_predictions = best_svd.test(testset)
knn_predictions = best_knn.test(testset)

svd_rmse = accuracy.rmse(svd_predictions, verbose=False)
svd_mae = accuracy.mae(svd_predictions, verbose=False)

knn_rmse = accuracy.rmse(knn_predictions, verbose=False)
knn_mae = accuracy.mae(knn_predictions, verbose=False)

print(f"SVD      -> Test RMSE: {svd_rmse:.4f}, Test MAE: {svd_mae:.4f}")
print(f"KNNBasic -> Test RMSE: {knn_rmse:.4f}, Test MAE: {knn_mae:.4f}")

SVD      -> Test RMSE: 0.9192, Test MAE: 0.7234
KNNBasic -> Test RMSE: 1.0120, Test MAE: 0.7984


In [13]:
results = pd.DataFrame([
    {"Model": "SVD", "RMSE": svd_rmse, "MAE": svd_mae},
    {"Model": "KNNBasic", "RMSE": knn_rmse, "MAE": knn_mae},
]).sort_values(by="RMSE", ascending=True).reset_index(drop=True)

display(results)

,Model,RMSE,MAE
0,SVD,0.919175,0.723355
1,KNNBasic,1.011953,0.798421


**Таблиця тестових результатів:**

| Модель | Test RMSE | Test MAE |
|:--|--:|--:|
| SVD | ... | ... |
| KNNBasic | ... | ... |

**Порівняння з CV:** Значення RMSE та MAE на незалежній тестовій вибірці (testset) загалом узгоджуються з результатами крос-валідації (cv=3) на навчальній частині, що свідчить про відсутність сильного перенавчання та стабільну узагальнювальну здатність моделей.

## 8. Топ-10 рекомендацій для користувача

1. Оберіть модель із кращим **CV RMSE**.
2. Задайте `user_id` (вихідний приклад використовує `'196'`).
3. Отримайте фільми, які цей користувач уже оцінив у навчальній вибірці.
4. Для решти фільмів обчисліть прогнозований рейтинг `.predict(uid=..., iid=...).est`.
5. Виведіть **10 фільмів із найбільшим прогнозованим рейтингом**.

**Важливо:** не пропонуйте користувачу фільми, які він уже оцінив у навчальних даних. Для коректного перетворення внутрішніх та зовнішніх ID використовуйте `trainset.to_inner_uid()`, `trainset.to_raw_iid()`.

In [14]:
user_id = "196"

# 8.1. Визначаємо найкращу за CV RMSE модель
if grid_search_svd.best_score["rmse"] <= grid_search_knn.best_score["rmse"]:
    rec_model = best_svd
    rec_model_name = "SVD"
else:
    rec_model = best_knn
    rec_model_name = "KNNBasic"

print(f"Модель для рекомендацій (за кращим CV RMSE): {rec_model_name}")

# 8.2. Перевіряємо наявність користувача у trainset
try:
    inner_uid = trainset.to_inner_uid(str(user_id))
except ValueError:
    print(f"Користувача '{user_id}' немає у trainset. Використовуємо популярні фільми за середньою оцінкою з train_df.")
    top_popular = (
        train_df.groupby("item_id")["rating"]
        .mean()
        .sort_values(ascending=False)
        .head(10)
        .reset_index()
        .rename(columns={"rating": "predicted_rating"})
    )
    display(top_popular)
else:
    # 8.3. Отримуємо вже оцінені фільми у trainset та формуємо список неоцінених
    rated_inner_iids = {iid for (iid, _) in trainset.ur[inner_uid]}
    all_raw_iids = [trainset.to_raw_iid(iid) for iid in trainset.all_items() if iid not in rated_inner_iids]

    # 8.4. Прогнозуємо оцінки та виводимо топ-10
    preds = [
        (raw_iid, rec_model.predict(uid=str(user_id), iid=raw_iid).est)
        for raw_iid in all_raw_iids
    ]
    top10_recs = (
        pd.DataFrame(preds, columns=["item_id", "predicted_rating"])
        .sort_values(by="predicted_rating", ascending=False)
        .head(10)
        .reset_index(drop=True)
    )
    display(top10_recs)

Модель для рекомендацій (за кращим CV RMSE): SVD


,item_id,predicted_rating
0,320,4.490358
1,64,4.470680
2,480,4.413710
3,313,4.401920
4,513,4.399624
5,483,4.398092
6,318,4.392102
7,656,4.385379
8,22,4.377388
9,272,4.376010


## 9. Перевірка виконання

Позначте після реалізації:

- [ ] Дані `ml-100k` завантажено та перевірено.
- [ ] Є розбиття 80/20 з фіксованим `random_state`.
- [ ] `Reader`, `train_data` і `testset` створені коректно.
- [ ] GridSearchCV для **обох** моделей виконано лише на `train_data`.
- [ ] Для SVD і KNNBasic показано параметри та результати CV.
- [ ] Моделі навчено на повній навчальній частині.
- [ ] На незалежній тестовій вибірці розраховано **RMSE та MAE** для обох моделей.
- [ ] Наведено порівняльну таблицю та інтерпретацію.
- [ ] Сформовано топ-10 рекомендацій без уже оцінених фільмів.
- [ ] Усі комірки виконуються послідовно після перезапуску середовища.

## 10. У роботі на наборі даних MovieLens 100K (100 000 оцінок, 943 користувачі, 1 682 фільми) після розбиття 80/20 виконано підбір гіперпараметрів через GridSearchCV (cv=3) для алгоритмів SVD та KNNBasic. За результатами крос-валідації та тестування на незалежній тестовій вибірці модель SVD показала кращі (менші) значення RMSE та MAE порівняно з KNNBasic, тому саме її обрано для формування персональних рекомендацій. Підібрані гіперпараметри дозволили збалансувати точність та узагальнення моделей, а метрики на тесті виявилися близькими до показників крос-валідації. Для користувача '196' сформовано топ-10 рекомендацій виключно серед фільмів, які він ще не оцінював у навчальній вибірці. Водночас запропонований підхід має обмеження щодо нових користувачів і нових фільмів (проблема холодного старту), оскільки SVD і KNNBasic спираються на історію взаємодій у trainset.

Заповніть **власними результатами**:

1. Яку модель обрано за кросвалідацією і чому?
2. Які найкращі гіперпараметри отримано для обох моделей?
3. Які підсумкові значення RMSE та MAE на тестовій вибірці?
4. Чи є різниця між результатами CV та тестування?
5. Які обмеження мають отримані рекомендації (наприклад, нові користувачі/фільми)?

**Висновок:** _Напишіть 5–8 речень з опорою на отримані числові результати._

---
**Примітка до шаблону:** вихідний студентський ноутбук містив блок «Доцільна реалізація» з SVD, KNNBasic і топ-10 рекомендаціями. Тому тут відтворено ці етапи; додаткових офіційних вимог, не наведених у зразку, не припускаємо.

**Додаткове завдання. Написати власний рекомендаційний алгоритм з нуля**

In [15]:
class SimpleBaselineRecommender:
    """Простий рекомендаційний алгоритм на основі зміщення користувача та фільму (з регуляризацією)."""

    def __init__(self, reg_u=5.0, reg_i=10.0):
        self.reg_u = reg_u
        self.reg_i = reg_i
        self.global_mean = 0.0
        self.item_bias = {}
        self.user_bias = {}

    def fit(self, df_train):
        self.global_mean = float(df_train["rating"].mean())

        item_groups = df_train.groupby("item_id")["rating"].agg(["count", "sum"])
        for iid, row in item_groups.iterrows():
            self.item_bias[iid] = (row["sum"] - row["count"] * self.global_mean) / (row["count"] + self.reg_i)

        df_tmp = df_train.copy()
        df_tmp["res"] = df_tmp["rating"] - self.global_mean - df_tmp["item_id"].map(self.item_bias).fillna(0.0)
        user_groups = df_tmp.groupby("user_id")["res"].agg(["count", "sum"])
        for uid, row in user_groups.iterrows():
            self.user_bias[uid] = row["sum"] / (row["count"] + self.reg_u)
        return self

    def predict_one(self, uid, iid):
        est = self.global_mean + self.item_bias.get(iid, 0.0) + self.user_bias.get(uid, 0.0)
        return float(np.clip(est, 1.0, 5.0))

    def evaluate(self, df_test):
        preds = [self.predict_one(r.user_id, r.item_id) for r in df_test.itertuples(index=False)]
        y_true = df_test["rating"].values
        rmse = float(np.sqrt(np.mean((y_true - preds) ** 2)))
        mae = float(np.mean(np.abs(y_true - preds)))
        return rmse, mae

custom_rec = SimpleBaselineRecommender(reg_u=5.0, reg_i=10.0).fit(train_df)
c_rmse, c_mae = custom_rec.evaluate(test_df)
print(f"Власний алгоритм (Baseline Bias) -> Test RMSE: {c_rmse:.4f}, Test MAE: {c_mae:.4f}")

Власний алгоритм (Baseline Bias) -> Test RMSE: 0.9430, Test MAE: 0.7459
